# EXP-006 - Confirmatory: Full Aggregation Engine (winning-solution replication)

**Not a hypothesis** - a confirmatory replication of a *verified* published technique
(Deotte, `xgb-fraud-with-magic-0-9600.ipynb`; see `docs/kaggle/gap-analysis.md`). H1-H4 are
closed. This isolates **aggregation richness** by holding the model class fixed (LightGBM,
same params as EXP-001..004) and replacing EXP-004's 4-aggregate block with the full ~47-
feature engine. DeLong vs EXP-004 measures exactly the aggregation lever.

**Verified feature engine (replicated):**
- combine keys `card1_addr1`, `card1_addr1_P_emaildomain`
- coarse aggregations: mean/std of `TransactionAmt, D9, D11` at `card1 / card1_addr1 /
  card1_addr1_P_emaildomain`
- D1-based `uid` aggregations: mean/std of `TransactionAmt, D4, D9, D10, D15`; mean of
  `C1..C14` (except C3); mean of `M1..M9`; std of `C14`
- nunique at `uid` of `P_emaildomain, dist1, DT_M, id_02, cents, C13, V314, V127, V136,
  V309, V307, V320`
- `outsider15 = (|D1-D15|>3)`; frequency-encode all UIDs

**Deliberate deviation:** NaN left native (LightGBM handles it) instead of Deotte's
`fillna(-1)` - consistent with our native-NaN choice since EXP-001.

**Outputs:** `holdout_pred_exp006.csv`, `submission.csv`. DeLong computed off-notebook.

**Anchors:** EXP-004 (minimal UID) private 0.9032; Deotte single XGB w/ full engine private 0.9324.

In [ ]:
import os
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

SPLIT_QUANTILE = 0.8
SECONDS_PER_MONTH = 86400 * 30.44
EXCLUDE_COLS = {"isFraud", "TransactionID", "TransactionDT"}
MISSING_TOKEN = "__missing__"
FREQ_NUMERIC_CATS = ["card1", "card2", "card3", "card5", "addr1", "addr2"]
D_NORM_COLS = [f"D{i}" for i in range(1, 16) if i != 9]
GROUP_KEYS = ["card1_addr1", "card1_addr1_P_emaildomain", "uid"]

# --- verified aggregation spec (Deotte) ---
COARSE_KEYS = ["card1", "card1_addr1", "card1_addr1_P_emaildomain"]
COARSE_VALS = ["TransactionAmt", "D9", "D11"]
UID_MEANSTD_VALS = ["TransactionAmt", "D4", "D9", "D10", "D15"]
C_MEAN_COLS = [f"C{i}" for i in range(1, 15) if i != 3]
M_COLS = [f"M{i}" for i in range(1, 10)]
NUNIQUE_COLS = ["P_emaildomain", "dist1", "DT_M", "id_02", "cents",
                "C13", "V314", "V127", "V136", "V309", "V307", "V320"]

LGB_PARAMS = dict(
    objective="binary", learning_rate=0.05, num_leaves=192, min_data_in_leaf=100,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, seed=42, n_jobs=-1, verbosity=-1,
)
MAX_ROUNDS = 5000
ES_PATIENCE = 200

ON_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
if ON_KAGGLE:
    hits = sorted(Path("/kaggle/input").rglob("train_transaction.csv"))
    if not hits:
        raise FileNotFoundError("Competition data not attached (Add Input -> Competitions).")
    DATA_DIR = hits[0].parent
else:
    DATA_DIR = Path("../../../data/raw")
print(f"Data dir: {DATA_DIR}")

## 0. Feature functions - inline mirrors of `src/features/engineering.py`

In [ ]:
def _as_str(values):
    return values.astype("object").where(values.notna(), MISSING_TOKEN).astype(str)

def frequency_encode(train_values, values):
    freq = _as_str(train_values).value_counts(normalize=True)
    return _as_str(values).map(freq).fillna(0.0).astype("float32")

def label_encode(train_values, values):
    cats = {v: i for i, v in enumerate(sorted(_as_str(train_values).unique()))}
    return _as_str(values).map(cats).fillna(-1).astype("int32")

def split_email_domain(values, prefix):
    parts = _as_str(values).str.split(".")
    return pd.DataFrame(
        {f"{prefix}_provider": parts.str[0], f"{prefix}_suffix": parts.str[-1]},
        index=values.index,
    )

def build_categorical_block(train_df, df, label_cols, freq_cols):
    out = pd.DataFrame(index=df.index)
    for col in label_cols:
        out[f"{col}_le"] = label_encode(train_df[col], df[col])
    for col in freq_cols:
        out[f"{col}_freq"] = frequency_encode(train_df[col], df[col])
    return out

def add_time_features(transaction_dt):
    return pd.DataFrame(
        {"tx_hour": ((transaction_dt // 3600) % 24).astype("int32"),
         "tx_dow": ((transaction_dt // 86400) % 7).astype("int32")},
        index=transaction_dt.index,
    )

def add_amount_features(amount):
    cents = (amount - np.floor(amount)).round(2)
    return pd.DataFrame(
        {"amt_log1p": np.log1p(amount).astype("float32"), "amt_cents": cents.astype("float32")},
        index=amount.index,
    )

def normalize_d_columns(df, transaction_dt, d_cols):
    days = transaction_dt / 86400.0
    out = pd.DataFrame(index=df.index)
    for col in d_cols:
        out[f"{col}_norm"] = (df[col] - days).astype("float32")
    return out

def make_uid(df, card_col="card1", addr_col="addr1", dt_col="TransactionDT", d1_col="D1"):
    day = (df[dt_col] / 86400.0).round()
    ref = (day - df[d1_col]).round().astype("Int64").astype("string").fillna("NA")
    card = df[card_col].astype("Int64").astype("string").fillna("NA")
    addr = df[addr_col].astype("Int64").astype("string").fillna("NA")
    return (card + "_" + addr + "_" + ref).astype("object")

def combine_columns(df, col1, col2, sep="_"):
    return (_as_str(df[col1]) + sep + _as_str(df[col2])).astype("object")

def aggregate_group(df, group_key, value_cols, aggs=("mean", "std")):
    g = df.groupby(group_key)
    out = pd.DataFrame(index=df.index)
    for col in value_cols:
        for agg in aggs:
            out[f"{col}_{group_key}_{agg}"] = g[col].transform(agg).astype("float32")
    return out

def aggregate_nunique(df, group_key, value_cols):
    g = df.groupby(group_key)
    out = pd.DataFrame(index=df.index)
    for col in value_cols:
        out[f"{group_key}_{col}_ct"] = g[col].transform("nunique").astype("float32")
    return out

## 1. Load train + test, build keys and the aggregation engine over the union

All aggregations are label-free, so they are computed once over the train+test union. The
UID / combine keys are frequency-encoded (fit on train rows only); the M columns are
numeric-encoded (NaN preserved) purely as aggregation inputs.

In [ ]:
train_transaction = pd.read_csv(DATA_DIR / "train_transaction.csv")
train_identity = pd.read_csv(DATA_DIR / "train_identity.csv")
train = train_transaction.merge(train_identity, on="TransactionID", how="left")
del train_transaction, train_identity

test_transaction = pd.read_csv(DATA_DIR / "test_transaction.csv")
test_identity = pd.read_csv(DATA_DIR / "test_identity.csv")
test_identity.columns = [c.replace("id-", "id_") for c in test_identity.columns]
test = test_transaction.merge(test_identity, on="TransactionID", how="left")
del test_transaction, test_identity

n_train = len(train)
full = pd.concat([train, test], axis=0, ignore_index=True, sort=False)
del train, test

# keys
full["card1_addr1"] = combine_columns(full, "card1", "addr1")
full["card1_addr1_P_emaildomain"] = combine_columns(full, "card1_addr1", "P_emaildomain")
full["uid"] = make_uid(full).to_numpy()
full["DT_M"] = (full["TransactionDT"] / SECONDS_PER_MONTH).astype(int)
full["cents"] = (full["TransactionAmt"] - np.floor(full["TransactionAmt"])).round(2).astype("float32")

# M columns -> numeric (NaN preserved) as aggregation inputs only
m_num = {}
for c in M_COLS:
    codes, _ = pd.factorize(full[c])
    m_num[c] = np.where(full[c].isna().to_numpy(), np.nan, codes).astype("float32")
m_num = pd.DataFrame(m_num, index=full.index)
m_num["uid"] = full["uid"].to_numpy()
print(f"Union: {len(full):,} rows | UID unique: {full['uid'].nunique():,}")

In [ ]:
%%time
agg_frames = []
# coarse-key mean/std of Amt, D9, D11
for key in COARSE_KEYS:
    agg_frames.append(aggregate_group(full, key, COARSE_VALS, ("mean", "std")))
# uid mean/std of Amt + D cols
agg_frames.append(aggregate_group(full, "uid", UID_MEANSTD_VALS, ("mean", "std")))
# uid mean of C cols (except C3)
agg_frames.append(aggregate_group(full, "uid", C_MEAN_COLS, ("mean",)))
# uid std of C14
agg_frames.append(aggregate_group(full, "uid", ["C14"], ("std",)))
# uid mean of M cols (from numeric-encoded inputs)
agg_frames.append(aggregate_group(m_num, "uid", M_COLS, ("mean",)))
# uid nunique of categorical-ish cols
agg_frames.append(aggregate_nunique(full, "uid", NUNIQUE_COLS))
# outsider15 interaction
outsider = pd.DataFrame(
    {"outsider15": (np.abs(full["D1"] - full["D15"]) > 3).astype("float32")}, index=full.index
)
agg_frames.append(outsider)
agg_engine = pd.concat(agg_frames, axis=1)
del agg_frames, m_num
print(f"Aggregation engine: {agg_engine.shape[1]} new features")

In [ ]:
# email splits (row-local)
for col, prefix in [("P_emaildomain", "P_email"), ("R_emaildomain", "R_email")]:
    full = pd.concat([full, split_email_domain(full[col], prefix)], axis=1)

numeric_features = [
    c for c in full.columns
    if full[c].dtype != "O" and c not in EXCLUDE_COLS and c not in ("DT_M",)
]
label_cols = [c for c in full.columns if full[c].dtype == "O" and c not in GROUP_KEYS]
freq_cols = label_cols + FREQ_NUMERIC_CATS + GROUP_KEYS  # UIDs freq-encoded, not label-encoded

row_local = pd.concat(
    [
        add_time_features(full["TransactionDT"]),
        add_amount_features(full["TransactionAmt"]),
        normalize_d_columns(full, full["TransactionDT"], D_NORM_COLS),
        agg_engine,
    ],
    axis=1,
)
X_num_full = pd.concat([full[numeric_features].astype("float32"), row_local], axis=1)
cat_source_full = full[label_cols + FREQ_NUMERIC_CATS + GROUP_KEYS].copy()
y = full["isFraud"].to_numpy()
dt = full["TransactionDT"].to_numpy()
months = full["DT_M"].to_numpy()
trans_ids = full["TransactionID"].to_numpy()
del full, row_local, agg_engine

is_train = np.arange(len(X_num_full)) < n_train
cutoff = np.quantile(dt[is_train], SPLIT_QUANTILE)
train_mask = is_train & (dt < cutoff)
holdout_mask = is_train & (dt >= cutoff)
print(f"Features: {X_num_full.shape[1]} | train {train_mask.sum():,} | holdout {holdout_mask.sum():,} | test {(~is_train).sum():,}")

def make_X(fit_mask, rows_mask):
    block = build_categorical_block(
        cat_source_full[fit_mask], cat_source_full[rows_mask], label_cols, freq_cols
    )
    return pd.concat([X_num_full[rows_mask].reset_index(drop=True), block.reset_index(drop=True)], axis=1)

## 2. Scheme B - month-wise GroupKFold (7 folds)

In [ ]:
train_months_all = months[is_train]
fold_aucs, fold_best_iters = [], []
t0 = time.time()
for m in sorted(set(train_months_all)):
    tr = is_train & (months != m)
    va = is_train & (months == m)
    X_tr = make_X(fit_mask=tr, rows_mask=tr)
    X_va = make_X(fit_mask=tr, rows_mask=va)
    clf = lgb.LGBMClassifier(n_estimators=MAX_ROUNDS, **LGB_PARAMS)
    clf.fit(
        X_tr, y[tr].astype(int),
        eval_set=[(X_va, y[va].astype(int))], eval_metric="auc",
        callbacks=[lgb.early_stopping(ES_PATIENCE, verbose=False), lgb.log_evaluation(0)],
    )
    auc = roc_auc_score(y[va].astype(int), clf.predict_proba(X_va)[:, 1])
    fold_aucs.append(auc); fold_best_iters.append(clf.best_iteration_)
    del X_tr, X_va
    print(f"fold month={m}: AUC={auc:.4f}  best_iter={clf.best_iteration_}  ({(time.time()-t0)/60:.1f} min)")

scheme_b_mean, scheme_b_std = float(np.mean(fold_aucs)), float(np.std(fold_aucs))
print(f"\nScheme B GroupKFold: {scheme_b_mean:.4f} +/- {scheme_b_std:.4f}")

## 3. Scheme A model - ES inside the train partition, refit at best_iter * 1.1

In [ ]:
es_month = months[train_mask].max()
sub_tr = train_mask & (months < es_month)
es_va = train_mask & (months == es_month)
assert es_va.sum() > 0 and sub_tr.sum() > 0, "empty ES split"

X_sub_tr = make_X(fit_mask=train_mask, rows_mask=sub_tr)
X_es_va = make_X(fit_mask=train_mask, rows_mask=es_va)
es_clf = lgb.LGBMClassifier(n_estimators=MAX_ROUNDS, **LGB_PARAMS)
es_clf.fit(
    X_sub_tr, y[sub_tr].astype(int),
    eval_set=[(X_es_va, y[es_va].astype(int))], eval_metric="auc",
    callbacks=[lgb.early_stopping(ES_PATIENCE, verbose=False), lgb.log_evaluation(0)],
)
best_iter = es_clf.best_iteration_
final_rounds = max(int(best_iter * 1.1), 100)
del X_sub_tr, X_es_va
print(f"ES month: {es_month} | best_iter: {best_iter} | refit rounds: {final_rounds}")

X_train = make_X(fit_mask=train_mask, rows_mask=train_mask)
model = lgb.LGBMClassifier(n_estimators=final_rounds, **LGB_PARAMS)
model.fit(X_train, y[train_mask].astype(int))
feature_cols = list(X_train.columns)
del X_train

X_holdout = make_X(fit_mask=train_mask, rows_mask=holdout_mask)
val_proba = model.predict_proba(X_holdout)[:, 1]
holdout_auc = roc_auc_score(y[holdout_mask].astype(int), val_proba)
del X_holdout
print(f"Scheme A holdout ROC-AUC: {holdout_auc:.4f}  (EXP-004: 0.9299)")

## 4. Save holdout predictions and submission

In [ ]:
pd.DataFrame(
    {"TransactionID": trans_ids[holdout_mask], "y_true": y[holdout_mask].astype(int), "score": val_proba}
).to_csv("holdout_pred_exp006.csv", index=False)
print("Saved holdout_pred_exp006.csv")

X_test = make_X(fit_mask=train_mask, rows_mask=~is_train)
assert list(X_test.columns) == feature_cols, "feature contract violated"
test_proba = model.predict_proba(X_test)[:, 1]
pd.DataFrame(
    {"TransactionID": trans_ids[~is_train].astype(int), "isFraud": test_proba}
).to_csv("submission.csv", index=False)
print(f"Saved submission.csv ({(~is_train).sum():,} rows, expected 506,691)")

print("\n=== EXP-006 summary ===")
print(f"Scheme A holdout AUC : {holdout_auc:.4f}")
print(f"Scheme B GroupKFold  : {scheme_b_mean:.4f} +/- {scheme_b_std:.4f}")
print(f"Per-fold AUCs        : {[round(float(a), 4) for a in fold_aucs]}")